# 05 — Comparação A × B em imagem inteira

Objetivo: comparar mesmos IDs com correspondência um a um e contabilizar parasitas não localizados. Entradas: manifesto, referência revisada e resultados A/B. Saída: relatório JSON pareado. Pré-requisitos: etapas anteriores. No modo sintético as previsões são artificiais para testar o avaliador; valores não representam desempenho dos sistemas.

Execução: na raiz, `python -m jupyter lab notebooks`. Execute de cima para baixo. Variáveis: `TCC_ROOT`, `TCC_RUN_MODE=synthetic|real`, `TCC_MANIFEST`, `TCC_ANNOTATIONS`, `TCC_SPLIT`. Colab pode definir `TCC_ROOT` para a cópia do repositório e instalar as dependências em runtime Python compatível.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("TCC_ROOT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Defina TCC_ROOT para a raiz do repositório."
sys.path.insert(0, str(ROOT))
MODE = os.environ.get("TCC_RUN_MODE", "synthetic")
assert MODE in {"synthetic", "real"}
from scripts.notebook_support import prepare_synthetic, SyntheticClassifier
from src import data_protocol as dp
if MODE == "synthetic":
    OUTPUT, rows = prepare_synthetic(ROOT)
    MANIFEST = OUTPUT / "images.csv"
    ANNOTATIONS = OUTPUT / "synthetic_annotations.json"
else:
    OUTPUT = ROOT / "experiments" / "runs" / "notebook_real"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    MANIFEST = Path(os.environ.get("TCC_MANIFEST", ROOT / "data/splits/trypomastigote/images.csv"))
    ANNOTATIONS = Path(os.environ.get("TCC_ANNOTATIONS", ROOT / "data/annotations/trypomastigote_pending.json"))
if MODE == "real" and not MANIFEST.is_file():
    raise FileNotFoundError("Partições reais ainda não preparadas. Siga docs/guia_treinamento.md, seção 4, e defina TCC_MANIFEST e TCC_ANNOTATIONS.")
print({"modo": MODE, "raiz": str(ROOT), "manifesto": str(MANIFEST)})


In [ ]:
if MODE == "synthetic":
    from scripts.notebook_support import synthetic_comparison
    fixture = synthetic_comparison(OUTPUT, rows)
    pred_a, pred_b = OUTPUT / "fixture_A.json", OUTPUT / "fixture_B.json"
    split = "test"
else:
    assert os.environ.get("TCC_PREDICTIONS_A") and os.environ.get("TCC_PREDICTIONS_B"), "Defina os dois caminhos de previsões."
    pred_a, pred_b = Path(os.environ["TCC_PREDICTIONS_A"]), Path(os.environ["TCC_PREDICTIONS_B"])
    split = os.environ.get("TCC_SPLIT", "test")
subprocess.run([sys.executable, str(ROOT / "scripts/compare_systems.py"),
                "--manifest", str(MANIFEST), "--annotations", str(ANNOTATIONS),
                "--predictions-a", str(pred_a), "--predictions-b", str(pred_b),
                "--split", split, "--output", str(OUTPUT / "comparison_cli.json")] + (["--synthetic-fixture"] if MODE=="synthetic" else []), check=True, cwd=ROOT)
report = json.loads((OUTPUT / "comparison_cli.json").read_text(encoding="utf-8"))
print({"modo":MODE,"imagens":len(report["image_ids"]),"grupos":report["n_groups"],"matching":report["matching"]})


Sem referência revisada, métricas supervisionadas permanecem indisponíveis. Erros não viram negativos e não removem imagens. Não usar recortes ou sementes como amostras independentes; intervalos por grupo dependem de grupos biológicos confirmados e quantidade suficiente.